# E5c v3 — overlap between E1 methods at k=50

Analysis-only: reads completed E1 selected panels; no feature selection is rerun.

In [ ]:
from pathlib import Path
import itertools, json, shutil
import numpy as np
import pandas as pd

# Kaggle: attach the completed E1 feature-selection ZIP and set this input path.
E1_ROOT = Path('/kaggle/input/e1-feature-selection/E1_feature_selection_v3')
OUT = Path('/kaggle/working/E5c_overlap'); OUT.mkdir(parents=True, exist_ok=True)
K = 50

def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else np.nan

rows=[]
for family_dir in sorted(E1_ROOT.iterdir()):
    if not family_dir.is_dir(): continue
    for cohort_dir in sorted(family_dir.iterdir()):
        f=cohort_dir/'selection_per_fold.csv'
        if not f.is_file(): continue
        d=pd.read_csv(f); d=d[d.k.eq(K)]
        for (repeat,fold), g in d.groupby(['repeat','fold']):
            panels={m:set(x.gene) for m,x in g.groupby('method')}
            for left,right in itertools.combinations(sorted(panels),2):
                rows.append({'cohort':cohort_dir.name,'family':family_dir.name,'repeat':repeat,'fold':fold,'k':K,'method_a':left,'method_b':right,'jaccard':jaccard(panels[left],panels[right])})
per_fold=pd.DataFrame(rows)
assert not per_fold.empty, 'No E1 k=50 selection panels found.'
summary=(per_fold.groupby(['cohort','family','k','method_a','method_b'],as_index=False).jaccard.agg(['mean','std','count']).reset_index().rename(columns={'mean':'jaccard_mean','std':'jaccard_sd','count':'n_outer_splits'}))
per_fold.to_csv(OUT/'e5c_overlap_k50_per_fold.csv',index=False)
summary.to_csv(OUT/'e5c_overlap_k50.csv',index=False)
(OUT/'config.json').write_text(json.dumps({'experiment':'E5c_overlap_v3','k':K,'source':str(E1_ROOT),'traceable':True},indent=2))
shutil.make_archive('/kaggle/working/e5c_overlap_v3','zip','/kaggle/working','E5c_overlap')
print('DONE:', '/kaggle/working/e5c_overlap_v3.zip', len(summary))